# Qwen3 + Solar Merge Ensemble Pipeline

## 구성
1. **Solar**: Embedding Retrieval Few-shot (intfloat/multilingual-e5-base)
2. **Qwen3**: LoRA SFT 기반 요약
3. **Merge Ensemble**: Solar가 두 요약을 합쳐 최종 생성

## 실행 순서
- Cell 1~4: 환경 설정
- Cell 5~8: Solar 설정 (Embedding + Prompt)
- Cell 9~13: Qwen3 설정 (LoRA SFT)
- Cell 14~16: Merge Ensemble + 최종 제출


## 1. 패키지 설치

In [1]:
# 처음 한 번만 실행
!pip install sentence-transformers scikit-learn openai rouge python-mecab-ko -q
!pip install unsloth peft trl datasets -q
!apt-get install -y mecab mecab-ipadic-utf8 libmecab-dev -q


Reading package lists...
Building dependency tree...
Reading state information...
libmecab-dev is already the newest version (0.996-10build1).
mecab-ipadic-utf8 is already the newest version (2.7.0-20070801+main-2.1).
mecab is already the newest version (0.996-10build1).
0 upgraded, 0 newly installed, 0 to remove and 46 not upgraded.


## 2. Import

In [2]:
import os, re, time, random
import pandas as pd
import numpy as np
import torch
from tqdm.auto import tqdm
from rouge import Rouge
from mecab import MeCab

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : NVIDIA GeForce RTX 3090
VRAM    : 25.4 GB


## 3. 경로 & API 설정

In [3]:
# ============================================================
# ⭐ 환경에 맞게 수정하세요
# ============================================================
DATA_PATH   = "/data/ephemeral/home/data/"
RESULT_PATH = "/data/ephemeral/home/code/prediction/"
os.makedirs(RESULT_PATH, exist_ok=True)

# Solar API
SOLAR_API_KEY = "up_opnjqOc0T694rx1It5hDPhfZNCiW5"
client = OpenAI(
    api_key=SOLAR_API_KEY,
    base_url="https://api.upstage.ai/v1/solar"
)

# Qwen3 모델 경로 (학습 완료된 LoRA 어댑터)
QWEN_MODEL_NAME = "unsloth/Qwen3-8B"       # base 모델
QWEN_LORA_PATH  = None                      # LoRA 완료 후 경로 입력
                                            # 예: "./outputs/qwen3_9b_lora_sft/lora_adapter"
MAX_SEQ_LENGTH  = 1024
MAX_NEW_TOKENS  = 128

print("설정 완료")


설정 완료


## 4. 데이터 로드

In [4]:
train_df = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
dev_df   = pd.read_csv(os.path.join(DATA_PATH, "dev.csv"))
test_df  = pd.read_csv(os.path.join(DATA_PATH, "test.csv"))

print(f"Train : {len(train_df):,}개")
print(f"Dev   : {len(dev_df):,}개")
print(f"Test  : {len(test_df):,}개")

# 태그 vs 이름 비율 확인 (EDA)
has_tag  = train_df['summary'].str.contains(r'#Person\d+#', regex=True).mean()
print(f"\n요약 #PersonN# 태그 비율: {has_tag:.1%}  ← 82%+ 이면 태그 유지 전략 유효")


Train : 12,457개
Dev   : 499개
Test  : 499개

요약 #PersonN# 태그 비율: 85.9%  ← 82%+ 이면 태그 유지 전략 유효


---
## ☀️ Solar 설정 (Embedding Retrieval Few-shot)


## 5. Embedding 인덱스 구축 (최초 1회, 약 5~10분)

In [5]:
# intfloat/multilingual-e5-base
# - e5 모델은 "query: " / "passage: " prefix 필수
# - topic 기반보다 의미 유사도가 높아 retrieval 정확도 향상
# - CPU 실행 → VRAM 추가 사용 없음

embed_model = SentenceTransformer(
    "intfloat/multilingual-e5-base",
    device="cpu"
)

print("Train dialogue embedding 인코딩 시작...")
train_embeddings = embed_model.encode(
    ["passage: " + str(d) for d in train_df["dialogue"]],
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)
print(f"✅ Embedding 인덱스 구축 완료: {train_embeddings.shape}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Train dialogue embedding 인코딩 시작...


Batches:   0%|          | 0/195 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 6. Retrieval 함수

In [ ]:
def get_few_shots_by_embedding(dialogue, n=3):
    """
    Embedding 유사도로 train에서 few-shot 선택
    - top5 뽑고 그 중 3개 랜덤 선택 → 다양성 증가
    - topic 예측 오류 없음, 대화 내용 자체로 매칭
    """
    query_vec = embed_model.encode(
        ["query: " + str(dialogue)],
        convert_to_numpy=True
    )
    sims    = cosine_similarity(query_vec, train_embeddings).flatten()
    top5    = sims.argsort()[::-1][:5]
    selected = np.random.choice(top5, min(n, len(top5)), replace=False)
    return train_df.iloc[selected].to_dict("records")


# 테스트
sample = train_df.iloc[0]["dialogue"]
shots  = get_few_shots_by_embedding(sample)
print(f"retrieval 테스트: {len(shots)}개 반환")
print(f"예시 요약: {shots[0]['summary']}")


## 7. Solar Prompt & 요약 함수

In [ ]:
# ============================================================
# System Prompt - 압축 중심 (v7)
# ============================================================
SOLAR_SYSTEM_PROMPT = (
    "당신은 한국어 대화 요약 전문가입니다.\n\n"
    "주어진 대화의 핵심 사건을 한국어 문어체로 1문장으로 요약하세요.\n\n"
    "규칙:\n"
    "1. 반드시 1문장으로만 작성합니다.\n"
    "2. 20단어 이하로 작성합니다.\n"
    "3. 대화의 가장 핵심 사건 1개만 포함합니다.\n"
    "4. 세부 설명, 이유, 결과, 추가 정보 등 불필요한 내용은 포함하지 않습니다.\n"
    "5. #Person1#, #Person2# 등 화자 태그는 절대 변경하지 않습니다.\n"
    "6. 이름이 대화에 명시적으로 등장한 경우에만 이름을 사용합니다.\n\n"
    "요약 형태 (다음 중 하나를 따르세요):\n"
    "- #Person1#이 #Person2#에게 ~을 제안한다.\n"
    "- #Person1#은 #Person2#에게 ~을 요청한다.\n"
    "- #Person1#과 #Person2#는 ~에 대해 이야기한다.\n"
    "- #Person1#이 #Person2#에게 ~을 묻는다.\n"
    "- #Person1#이 #Person2#에게 ~을 설명한다.\n\n"
    "대화 전체를 설명하지 말고 핵심 행동만 요약하세요."
)


def build_solar_prompt(dialogue):
    """Embedding retrieval few-shot 포함 프롬프트 구성"""
    few_shots = get_few_shots_by_embedding(dialogue, n=3)

    # system
    messages = [{"role": "system", "content": SOLAR_SYSTEM_PROMPT}]

    # few-shot examples (user/assistant 교차)
    for ex in few_shots:
        messages.append({"role": "user",      "content": f"대화:\n{ex['dialogue']}"})
        messages.append({"role": "assistant", "content": ex["summary"]})

    # target dialogue
    messages.append({
        "role": "user",
        "content": (
            f"대화:\n{dialogue}\n\n"
            "위 대화를 #Person 태그를 그대로 유지하여 반드시 1문장, 20단어 이하로 요약하세요:\n요약:"
        )
    })
    return messages


def solar_summarize(dialogue, max_retries=3):
    """Solar API 요약 - 압축 중심, 길이 제한 후처리 포함"""
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="solar-1-mini-chat",
                messages=build_solar_prompt(dialogue),
                temperature=0.2,   # 낮게 → 압축적 생성
                top_p=0.9,
                max_tokens=60      # 생성 길이 강제 제한
            )
            result = response.choices[0].message.content.strip()

            # 후처리: 20단어 초과 시 자르기
            words = result.split()
            if len(words) > 20:
                result = " ".join(words[:20])

            return result
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep((attempt + 1) * 5)
            else:
                print(f"Solar 실패: {e}")
                return ""


# 빠른 검증
print("[Solar 빠른 테스트]")
test_dialogue = dev_df.iloc[0]["dialogue"]
result = solar_summarize(test_dialogue)
print(f"예측: {result}")
print(f"정답: {dev_df.iloc[0]['summary']}")


## 8. Solar Dev 검증 (n=10 빠른 확인)

In [ ]:
def validate_solar(num_samples=10):
    """Solar 빠른 검증 - MeCab 기반 ROUGE"""
    rouge  = Rouge()
    m      = MeCab()

    def tok(text):
        tokens = [t for t, _ in m.pos(str(text)) if t.strip()]
        return " ".join(tokens) if tokens else str(text)

    samples = dev_df.sample(num_samples, random_state=42).reset_index(drop=True)
    results = []

    for i, row in tqdm(samples.iterrows(), total=num_samples, desc="Solar validate"):
        pred = solar_summarize(row["dialogue"])
        gold = str(row["summary"]).strip()
        if not pred.strip() or not gold.strip():
            continue
        try:
            s   = rouge.get_scores(tok(pred), tok(gold))[0]
            avg = (s["rouge-1"]["f"] + s["rouge-2"]["f"] + s["rouge-l"]["f"]) / 3
            results.append(avg)
            print(f"[{i}] AVG={avg:.3f} | 예측: {pred}")
            print(f"      정답: {gold}")
        except:
            pass

    print(f"\n{'='*50}")
    print(f"Solar Validate ROUGE AVG: {np.mean(results):.4f}  ({num_samples}샘플)")
    print(f"{'='*50}")
    return np.mean(results)

validate_solar(10)


---
## 🤖 Qwen3 설정 (LoRA SFT)


## 9. Qwen3 하이퍼파라미터

In [ ]:
# ============================================================
# ⭐ RTX 3090 (24GB) 안정 설정
# ============================================================
EXP_NAME         = "qwen3_9b_lora_sft_v2"
QWEN_MODEL_NAME  = "unsloth/Qwen3-8B"   # ⭐ 실제 사용 모델명으로 수정
MAX_SEQ_LENGTH   = 1024   # 2048 → 1024 (메모리 절약)
LORA_R           = 16     # 32 → 16
LORA_ALPHA       = 16
LORA_DROPOUT     = 0.0
LEARNING_RATE    = 2e-4
EPOCHS           = 3
WARMUP_RATIO     = 0.05
WEIGHT_DECAY     = 0.01
PER_DEVICE_BATCH = 1      # 2 → 1
GRAD_ACCUM       = 8      # 16 → 8  (effective batch=8)
MAX_NEW_TOKENS   = 128    # 192 → 128

# 메모리 최적화
torch.cuda.empty_cache()
torch.backends.cuda.matmul.allow_tf32 = True

print(f"Effective batch size: {PER_DEVICE_BATCH * GRAD_ACCUM}")
print(f"남은 VRAM: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1e9:.1f} GB")


## 10. Qwen3 모델 로드 + LoRA 설정

In [ ]:
import os
os.environ["UNSLOTH_DISABLE_TRITON"] = "1"   # triton 컴파일러 없는 환경용

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=QWEN_MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,
)
print(f"모델 로드 완료: {QWEN_MODEL_NAME}")

torch.cuda.empty_cache()

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"학습 파라미터: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")


## 11. Qwen3 SFT 학습

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
from dataclasses import dataclass
from typing import Any, Dict, List

# ── System Prompt (압축 중심) ──────────────────────────
QWEN_SYSTEM_PROMPT = (
    "당신은 한국어 대화 요약 전문가입니다.\n\n"
    "규칙:\n"
    "1. 반드시 1문장으로만 작성합니다.\n"
    "2. 20단어 이하로 작성합니다.\n"
    "3. 대화의 가장 핵심 사건 1개만 포함합니다.\n"
    "4. 불필요한 세부 설명은 포함하지 않습니다.\n"
    "5. #Person1#, #Person2# 등 화자 태그는 절대 변경하지 않습니다.\n\n"
    "요약 형태:\n"
    "- #Person1#이 #Person2#에게 ~을 제안한다.\n"
    "- #Person1#은 #Person2#에게 ~을 요청한다.\n"
    "- #Person1#과 #Person2#는 ~에 대해 이야기한다."
)

def create_messages(dialogue, summary=None):
    msgs = [
        {"role": "system", "content": QWEN_SYSTEM_PROMPT},
        {"role": "user",   "content": f"아래 대화를 1문장, 20단어 이하로 요약하세요:\n\n{dialogue}"},
    ]
    if summary is not None:
        msgs.append({"role": "assistant", "content": str(summary)})
    return msgs

def formatting_func(examples):
    texts = []
    for d, s in zip(examples["dialogue"], examples["summary"]):
        text = tokenizer.apply_chat_template(
            create_messages(d, s),
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False
        )
        texts.append(text)
    return {"text": texts}

train_dataset = Dataset.from_pandas(train_df[["dialogue","summary"]]).map(formatting_func, batched=True)
dev_dataset   = Dataset.from_pandas(dev_df[["dialogue","summary"]]).map(formatting_func,   batched=True)
print(f"학습: {len(train_dataset):,}개 | 검증: {len(dev_dataset):,}개")

# ── Response-Only DataCollator ─────────────────────────
@dataclass
class ResponseOnlyCollator:
    tokenizer: Any
    response_template_ids: List[int] = None
    max_length: int = 1024

    def __call__(self, features):
        if "input_ids" in features[0]:
            batch = self.tokenizer.pad(features, padding=True, max_length=self.max_length, return_tensors="pt")
        else:
            texts = [f["text"] for f in features]
            batch = self.tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=self.max_length)
        labels = batch["input_ids"].clone()
        for i in range(len(labels)):
            ids  = batch["input_ids"][i].tolist()
            pos  = self._find_start(ids)
            if pos >= 0:
                labels[i, :pos] = -100
            labels[i, batch["attention_mask"][i] == 0] = -100
        batch["labels"] = labels
        return batch

    def _find_start(self, ids):
        tmpl = self.response_template_ids
        if not tmpl: return 0
        last = -1
        for i in range(len(ids) - len(tmpl) + 1):
            if ids[i:i+len(tmpl)] == tmpl:
                last = i + len(tmpl)
        return last

resp_tmpl_ids = tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)
collator      = ResponseOnlyCollator(tokenizer=tokenizer, response_template_ids=resp_tmpl_ids, max_length=MAX_SEQ_LENGTH)

# ── SFTTrainer ─────────────────────────────────────────
OUTPUT_DIR = f"/data/ephemeral/home/code/outputs/{EXP_NAME}"

sft_config = SFTConfig(
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    seed=42,
    output_dir=OUTPUT_DIR,
    report_to="wandb",
    optim="adamw_8bit",
    max_grad_norm=1.0,
    run_name=EXP_NAME,
    dataloader_num_workers=0,
)

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=train_dataset, eval_dataset=dev_dataset,
    args=sft_config, data_collator=collator,
)

print(f"학습 시작: effective batch={PER_DEVICE_BATCH*GRAD_ACCUM}, BF16={is_bfloat16_supported()}")
trainer_stats = trainer.train()
print(f"\n✅ 학습 완료! loss={trainer_stats.metrics['train_loss']:.4f}")

# LoRA 저장
LORA_SAVE_PATH = os.path.join(OUTPUT_DIR, "lora_adapter")
model.save_pretrained(LORA_SAVE_PATH)
tokenizer.save_pretrained(LORA_SAVE_PATH)
print(f"✅ LoRA 저장: {LORA_SAVE_PATH}")
QWEN_LORA_PATH = LORA_SAVE_PATH   # 이후 셀에서 사용


## 12. Qwen3 추론 함수

In [ ]:
# LoRA 학습 완료 후 실행
FastLanguageModel.for_inference(model)


def postprocess(text):
    """생성 후처리"""
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"#\s*Person\s*(\d+)\s*#", r"#Person\1#", text)
    text = re.sub(r"^요약\s*:\s*", "", text).strip()
    text = re.sub(r"\s+", " ", text).strip()
    # 20단어 초과 후처리
    words = text.split()
    if len(words) > 20:
        text = " ".join(words[:20])
    return text if text else "빈 요약"


def qwen_summarize(dialogue):
    """Qwen3 LoRA 요약 생성"""
    messages = create_messages(dialogue)   # summary 없이 → 추론 모드
    text = tokenizer.apply_chat_template(
        messages, tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )
    inputs = tokenizer(
        text, return_tensors="pt",
        truncation=True, max_length=MAX_SEQ_LENGTH
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=1.0,    # do_sample=False면 무시됨
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    result    = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return postprocess(result)


# 추론 예시 확인
print("[Qwen3 추론 예시]")
for i in range(3):
    pred = qwen_summarize(dev_df.iloc[i]["dialogue"])
    gold = dev_df.iloc[i]["summary"]
    print(f"\n[{i}] 정답: {gold}")
    print(f"     예측: {pred}")


## 13. Qwen3 Dev ROUGE 평가

In [ ]:
print(f"Dev 추론 시작 ({len(dev_df)}개)...")
dev_preds = [qwen_summarize(dev_df.iloc[i]["dialogue"]) for i in tqdm(range(len(dev_df)))]

rouge = Rouge()
m     = MeCab()

def tok(text):
    tokens = [t for t, _ in m.pos(str(text)) if t.strip()]
    return " ".join(tokens) if tokens else str(text)

golds     = [str(s).strip() for s in dev_df["summary"]]
preds_tok = [tok(p) for p in dev_preds]
golds_tok = [tok(g) for g in golds]

scores = rouge.get_scores(preds_tok, golds_tok, avg=True)
r1, r2, rl = scores["rouge-1"]["f"], scores["rouge-2"]["f"], scores["rouge-l"]["f"]
avg = (r1+r2+rl)/3

print(f"\n{'='*50}")
print(f"[Qwen3] Dev ROUGE (MeCab 기반)")
print(f"  ROUGE-1 : {r1:.4f}")
print(f"  ROUGE-2 : {r2:.4f}")
print(f"  ROUGE-L : {rl:.4f}")
print(f"  AVG     : {avg:.4f}")
print(f"{'='*50}")


---
## 🔀 Merge Ensemble (Qwen3 + Solar → Solar merge)


## 14. Merge Ensemble 함수

In [ ]:
def merge_ensemble(dialogue, qwen_summary, solar_summary):
    """
    Solar에게 Qwen3 + Solar 요약을 merge하도록 요청
    - Qwen3 = main (우선순위 높음)
    - Solar = 보조
    - 실패 시 Qwen3 결과 반환
    """
    merge_prompt = [
        {
            "role": "system",
            "content": (
                "두 요약을 참고하여 가장 정확한 최종 요약 1문장을 작성하세요.\n"
                "규칙:\n"
                "1. 반드시 1문장\n"
                "2. 20단어 이하\n"
                "3. #Person 태그 유지\n"
                "4. 핵심 사건만 포함\n"
                "5. 요약 1이 더 정확하면 요약 1을 기반으로 작성하세요."
            )
        },
        {
            "role": "user",
            "content": (
                f"대화:\n{dialogue}\n\n"
                f"요약 1 (주요 참고):\n{qwen_summary}\n\n"
                f"요약 2 (보조 참고):\n{solar_summary}\n\n"
                "최종 요약 (1문장, 20단어 이하):"
            )
        }
    ]
    try:
        response = client.chat.completions.create(
            model="solar-1-mini-chat",
            messages=merge_prompt,
            temperature=0.1,   # 매우 낮게 → 일관성
            max_tokens=60
        )
        result = response.choices[0].message.content.strip()
        words  = result.split()
        if len(words) > 20:
            result = " ".join(words[:20])
        return result
    except Exception as e:
        print(f"Merge 실패 → Qwen3 사용: {e}")
        return qwen_summary   # 실패 시 Qwen3 우선


# 빠른 테스트
sample_dialogue = dev_df.iloc[0]["dialogue"]
q = qwen_summarize(sample_dialogue)
s = solar_summarize(sample_dialogue)
m_result = merge_ensemble(sample_dialogue, q, s)

print(f"Qwen3  : {q}")
print(f"Solar  : {s}")
print(f"Merge  : {m_result}")
print(f"정답   : {dev_df.iloc[0]['summary']}")


## 15. Dev Merge ROUGE 비교 (Qwen3 vs Solar vs Merge)

In [ ]:
print("Dev Solar 추론 시작...")
dev_solar_preds = [solar_summarize(dev_df.iloc[i]["dialogue"]) for i in tqdm(range(len(dev_df)))]

print("\nDev Merge 추론 시작...")
dev_merge_preds = []
for i in tqdm(range(len(dev_df))):
    merged = merge_ensemble(
        dev_df.iloc[i]["dialogue"],
        dev_preds[i],           # Qwen3
        dev_solar_preds[i]      # Solar
    )
    dev_merge_preds.append(merged)


def eval_rouge(preds, golds, label):
    pt = [tok(p) for p in preds]
    gt = [tok(g) for g in golds]
    valid = [(p,g) for p,g in zip(pt,gt) if p.strip() and g.strip()]
    s   = rouge.get_scores([p for p,g in valid], [g for p,g in valid], avg=True)
    avg = (s["rouge-1"]["f"] + s["rouge-2"]["f"] + s["rouge-l"]["f"]) / 3
    print(f"[{label:<12}] R1={s['rouge-1']['f']:.4f}  R2={s['rouge-2']['f']:.4f}  RL={s['rouge-l']['f']:.4f}  AVG={avg:.4f}")
    return avg

print(f"\n{'='*60}")
print("📊 Dev ROUGE 비교 (MeCab 기반)")
print(f"{'='*60}")
q_avg = eval_rouge(dev_preds,        golds, "Qwen3")
s_avg = eval_rouge(dev_solar_preds,  golds, "Solar")
m_avg = eval_rouge(dev_merge_preds,  golds, "Merge")
print(f"{'='*60}")

best = max([("Qwen3",q_avg),("Solar",s_avg),("Merge",m_avg)], key=lambda x: x[1])
print(f"\n✅ 최고 모델: {best[0]} (AVG={best[1]:.4f})")


## 16. 최종 Test Inference & 제출 파일 생성

In [ ]:
# ============================================================
# ⭐ dev 결과 보고 best 전략 선택
# "merge" / "qwen3" / "solar"
# ============================================================
SUBMIT_STRATEGY = "merge"   # ← dev 결과 보고 변경

print(f"Test 추론 시작 ({len(test_df)}개) | 전략: {SUBMIT_STRATEGY}")
results = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    dialogue = str(row["dialogue"])

    if SUBMIT_STRATEGY == "qwen3":
        final = qwen_summarize(dialogue)

    elif SUBMIT_STRATEGY == "solar":
        final = solar_summarize(dialogue)

    else:  # merge
        q_sum = qwen_summarize(dialogue)
        s_sum = solar_summarize(dialogue)
        final = merge_ensemble(dialogue, q_sum, s_sum)

    results.append({"fname": row["fname"], "summary": final})

submission = pd.DataFrame(results)
save_path  = os.path.join(RESULT_PATH, f"submission_{SUBMIT_STRATEGY}_ensemble.csv")
submission.to_csv(save_path, index=False)

print(f"\n✅ 제출 파일 저장: {save_path}")
print(f"총 {len(submission)}개 | 고유 요약: {submission['summary'].nunique()}개")
print(f"평균 길이: {submission['summary'].str.split().str.len().mean():.1f}단어")
print(f"#Person 포함: {submission['summary'].str.contains('#Person').mean():.1%}")
print(f"\n[미리보기]")
print(submission.head(5).to_string())
